# Scientific Machine-Learning Workflow for Corrosion-Inhibition Modelling

This notebook is provided as a **code sample** to demonstrate a scientific-computing workflow used in materials-degradation research.

**Methods demonstrated:** PyTorch neural-network implementation, architecture search, regularisation, reproducible train/test splitting, explicit-equation extraction, model-performance evaluation, SHAP-based interpretability, physical-trend checks, and Williams applicability-domain analysis.

> **Data availability:** The underlying experimental dataset is associated with a manuscript currently under peer review and is not included in this code sample pending supervisor/journal approval. The notebook will execute when a correctly formatted local dataset is supplied.


## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import torch
import torch.nn as nn
import shap
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print('All imports successful.')

## 1. Load Data
**REPLACE** the synthetic block below with: `df = pd.read_excel('your_file.xlsx')` or `df = pd.read_csv('your_file.csv')`  

**Required column names (case-sensitive):**
- `Inhibitor_Concentration` — mg/mL of dried-extract equivalent (or your unit)
- `MEA_Concentration` — molarity (M)
- `Temperature` — kelvin (K)
- `Immersion_Time` — hours
- `pH` — dimensionless
- `IE` — inhibition efficiency (%)

In [ ]:
from pathlib import Path

DATA_PATH = Path("../data/ANN_dataset_cleaned.xlsx")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Research dataset not included in this application code sample. "
        "Place an authorised local copy at ../data/ANN_dataset_cleaned.xlsx."
    )

raw_df = pd.read_excel(DATA_PATH)
raw_df.columns = raw_df.columns.str.strip()

COLUMN_MAP = {
    "Inhibitor_Concentration (mL)": "Inhibitor_Concentration",
    "MEA_Concentration (wt%)": "MEA_Concentration",
    "Temperature (K)": "Temperature",
    "Immersion_Time (h)": "Immersion_Time",
    "pH": "pH",
    "IE (%)": "IE",
}

missing = [c for c in COLUMN_MAP if c not in raw_df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw_df[list(COLUMN_MAP)].rename(columns=COLUMN_MAP).copy()
for c in df.columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

if df.isna().any().any():
    bad = df.isna().sum()
    raise ValueError(f"Missing/non-numeric values detected:\n{bad[bad > 0]}")

INPUT_COLS = ["Inhibitor_Concentration", "MEA_Concentration", "Temperature", "Immersion_Time", "pH"]
OUTPUT_COL = "IE"
INPUT_LABELS = ["Inhibitor conc.", "MEA conc.", "Temperature", "Immersion time", "pH"]
INPUT_SYMBOLS = ["C_inh", "C_MEA", "T", "t", "pH"]

print(f"Dataset loaded successfully with {df.shape[0]} records and {df.shape[1]} variables.")
df.head()


## 2. Statistical Description → Paper Table 3
**Copy the printed output below into Paper Table 3.**

In [ ]:
stats = df.agg(['min','max','mean','median','std',
                 lambda x: x.max()-x.min(),
                 'skew',
                 lambda x: x.kurt()])
stats.index = ['Minimum','Maximum','Mean','Median','Std Dev','Range','Skewness','Kurtosis']

# Insert Mode row (compute robustly for continuous data)
mode_row = pd.Series({c: df[c].mode().iloc[0] if len(df[c].mode()) > 0 else np.nan
                       for c in df.columns}, name='Mode')
stats = pd.concat([stats.iloc[:4], pd.DataFrame([mode_row]), stats.iloc[4:]])

print('Statistical Description of Dataset (Paper Table 3)')
print('=' * 100)
print(stats.round(4).to_string())
print('=' * 100)

# Save for paper
stats.round(4).to_csv('paper_table3_statistics.csv')
print('\nSaved: paper_table3_statistics.csv')

## 3. Spearman Correlation Heatmap → Paper Figure 1

In [ ]:
corr_matrix, _ = spearmanr(df)
corr_df = pd.DataFrame(corr_matrix,
                        index=df.columns,
                        columns=df.columns)

mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f',
             cmap='RdBu_r', center=0, vmin=-1, vmax=1,
             linewidths=0.5, ax=ax,
             cbar_kws={'label': 'Spearman ρ'},
             square=True)
ax.set_title('Spearman correlation matrix — input variables and IE%',
              fontsize=12, pad=12)
plt.tight_layout()
plt.savefig('paper_fig1_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: paper_fig1_heatmap.png')

# Print correlation values explicitly for paper text
print('\nCorrelations with IE% (paper text reference):')
ie_corr = corr_df['IE'].drop('IE').sort_values(key=abs, ascending=False)
for name, val in ie_corr.items():
    direction = 'positive' if val > 0 else 'negative'
    strength = 'strong' if abs(val) > 0.6 else 'moderate' if abs(val) > 0.3 else 'weak'
    print(f'  {name:30s} ρ = {val:+.3f}  ({strength} {direction})')

## 4. Normalisation and Train/Test Split

In [ ]:
X = df[INPUT_COLS].values
y = df[OUTPUT_COL].values.reshape(-1, 1)

scaler_X = MinMaxScaler(feature_range=(-1, 1))
scaler_y = MinMaxScaler(feature_range=(-1, 1))

X_norm = scaler_X.fit_transform(X)
y_norm = scaler_y.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42)

Xtr = torch.FloatTensor(X_train)
ytr = torch.FloatTensor(y_train)
Xte = torch.FloatTensor(X_test)
yte = torch.FloatTensor(y_test)

print(f'Total samples    : {len(df)}')
print(f'Training samples : {X_train.shape[0]} (80%)')
print(f'Test samples     : {X_test.shape[0]} (20%)')

## 5. BRANN Class Definition (MacKay framework)

In [ ]:
class BRANN(nn.Module):
    """Feedforward MLP: 5 → n_hidden → 1"""
    def __init__(self, n_inputs=5, n_hidden=3, n_outputs=1):
        super().__init__()
        self.hidden = nn.Linear(n_inputs, n_hidden)
        self.output = nn.Linear(n_hidden, n_outputs)
        nn.init.xavier_uniform_(self.hidden.weight)
        nn.init.xavier_uniform_(self.output.weight)

    def forward(self, x):
        x = torch.tanh(self.hidden(x))
        x = self.output(x)
        return x


def train_brann(model, Xtr, ytr, epochs=1000, lr=0.01):
    """MacKay Bayesian Regularisation training loop."""
    n = Xtr.shape[0]
    k = sum(p.numel() for p in model.parameters())

    alpha = torch.tensor(0.001)
    beta  = torch.tensor(1.0)

    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(epochs):
        model.train()
        optimiser.zero_grad()

        pred = model(Xtr)
        SSE  = torch.sum((pred - ytr) ** 2)
        SSW  = sum(torch.sum(p ** 2) for p in model.parameters())

        F = beta * SSE + alpha * SSW
        F.backward()
        optimiser.step()

        if epoch % 10 == 0:
            with torch.no_grad():
                gamma = k * beta / (beta + alpha)
                alpha = gamma / (2.0 * SSW + 1e-10)
                beta  = (n - gamma) / (2.0 * SSE + 1e-10)
                alpha = alpha.clamp(1e-6, 1e3)
                beta  = beta.clamp(1e-6, 1e6)

        history.append((SSE / n).item())

    return history, alpha.item(), beta.item()


print('BRANN class and trainer defined.')

## 6. Architecture Search (2–10 neurons × 30 trials)

In [ ]:
NEURON_RANGE = range(2, 11)
N_TRIALS     = 30
EPOCHS       = 500

search_results = {}

print(f'Architecture search: 2–10 hidden neurons × {N_TRIALS} trials × {EPOCHS} epochs')
print('-' * 70)

for n_h in NEURON_RANGE:
    trial_test_mse = []
    for trial in range(N_TRIALS):
        torch.manual_seed(trial)
        model = BRANN(n_inputs=5, n_hidden=n_h)
        _, _, _ = train_brann(model, Xtr, ytr, epochs=EPOCHS)
        model.eval()
        with torch.no_grad():
            pred = model(Xte).numpy()
        test_mse = mean_squared_error(y_test, pred)
        trial_test_mse.append(test_mse)

    mean_mse = np.mean(trial_test_mse)
    best_mse = np.min(trial_test_mse)
    best_trial_idx = int(np.argmin(trial_test_mse))
    search_results[n_h] = {
        'mean_test_mse': mean_mse,
        'best_test_mse': best_mse,
        'best_trial':    best_trial_idx
    }
    print(f'  n_h = {n_h:2d}  |  mean test MSE = {mean_mse:.5f}  |  best = {best_mse:.5f}')

optimal_n_h = min(search_results, key=lambda k: search_results[k]['mean_test_mse'])
best_seed   = search_results[optimal_n_h]['best_trial']

print(f'\n>> OPTIMAL TOPOLOGY: 5 – {optimal_n_h} – 1 (best seed: {best_seed})')

## 7. Train Final Model with Optimal Topology

In [ ]:
torch.manual_seed(best_seed)
final_model = BRANN(n_inputs=5, n_hidden=optimal_n_h)
history, alpha_final, beta_final = train_brann(final_model, Xtr, ytr, epochs=1000)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history, color='#1D9E75', linewidth=1.2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Normalised MSE')
ax.set_yscale('log')
ax.set_title(f'BRANN training curve — topology 5 – {optimal_n_h} – 1')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('paper_fig_training_curve.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Final α = {alpha_final:.4f}')
print(f'Final β = {beta_final:.4f}')
print(f'Final MSE = {history[-1]:.6f}')

## 8. Weights, Biases, and Explicit Equation → Paper Table 6 and Eq. 6–9
Copy printed output directly into the paper.

In [ ]:
final_model.eval()
IW = final_model.hidden.weight.detach().numpy()   # (n_hidden, 5)
b1 = final_model.hidden.bias.detach().numpy()      # (n_hidden,)
LW = final_model.output.weight.detach().numpy()    # (1, n_hidden)
b2 = final_model.output.bias.detach().numpy()      # (1,)

# Table 6 equivalent
print('=' * 80)
print('PAPER TABLE 6 — BRANN Weights and Biases')
print('=' * 80)
header = f"{'Variable':<28}" + ''.join([f'  Neuron {j+1}'.rjust(12) for j in range(optimal_n_h)])
print(header)
print('-' * 80)
for i, name in enumerate(INPUT_LABELS):
    row = f"{name:<28}" + ''.join([f'  {IW[j,i]:+.6f}'.rjust(12) for j in range(optimal_n_h)])
    print(row)
print(f"{'b1 (hidden bias)':<28}" + ''.join([f'  {b1[j]:+.6f}'.rjust(12) for j in range(optimal_n_h)]))
print(f"{'LW (hidden → output)':<28}" + ''.join([f'  {LW[0,j]:+.6f}'.rjust(12) for j in range(optimal_n_h)]))
print(f"{'b2 (output bias)':<28}  {b2[0]:+.6f}")
print('=' * 80)

# Explicit equation
print('\nEXPLICIT EQUATION — copy directly into Paper Section 4.4')
print('=' * 80)
for j in range(optimal_n_h):
    terms = ' '.join([
        f'{"+" if IW[j,i] >= 0 else "-"} {abs(IW[j,i]):.4f}·{INPUT_SYMBOLS[i]}_n'
        for i in range(5)
    ]).lstrip('+ ').strip()
    bias_sign = '+' if b1[j] >= 0 else '-'
    print(f'A{j+1} = {LW[0,j]:+.4f} · tanh({terms} {bias_sign} {abs(b1[j]):.4f})')

sum_terms = ' + '.join([f'A{j+1}' for j in range(optimal_n_h)])
print(f'\nIE_n = {sum_terms} {"+" if b2[0] >= 0 else "-"} {abs(b2[0]):.4f}')

# Denormalisation
y_min = scaler_y.data_min_[0]
y_max = scaler_y.data_max_[0]
scale = (y_max - y_min) / 2
shift = (y_max + y_min) / 2
print(f'\nIE% (denormalised) = {scale:.4f} · IE_n + {shift:.4f}')

## 9. Physics-Informed Boundary Condition
Activates the explicit equation only when inhibitor concentration > 0.

In [ ]:
def physics_informed_IE(inhib_conc, mea_conc, temperature, immersion_time, pH):
    """
    Physics-informed prediction with boundary condition:
    IE% = 0 when inhibitor concentration = 0 (no inhibitor → no protection).
    """
    if inhib_conc == 0:
        return 0.0
    x_raw = np.array([[inhib_conc, mea_conc, temperature, immersion_time, pH]])
    x_norm = scaler_X.transform(x_raw)
    with torch.no_grad():
        ie_norm = final_model(torch.FloatTensor(x_norm)).numpy()
    ie = scaler_y.inverse_transform(ie_norm)[0, 0]
    return float(np.clip(ie, 0, 100))

# Verify
ie_zero  = physics_informed_IE(0,    3.0, 333, 12, 9.5)
ie_mid   = physics_informed_IE(0.25, 3.0, 333, 12, 9.5)
ie_high  = physics_informed_IE(0.50, 3.0, 333, 12, 9.5)
print(f'Boundary check:')
print(f'  IE% at C_inh = 0.00 mg/mL : {ie_zero:.2f}%  (physics constraint)')
print(f'  IE% at C_inh = 0.25 mg/mL : {ie_mid:.2f}%')
print(f'  IE% at C_inh = 0.50 mg/mL : {ie_high:.2f}%')

## 10. Performance Metrics → Paper Table 7

In [ ]:
def evaluate(X_tensor, y_norm, label):
    with torch.no_grad():
        pred_norm = final_model(X_tensor).numpy()
    pred = scaler_y.inverse_transform(pred_norm)
    true = scaler_y.inverse_transform(y_norm)
    return {
        'Label': label,
        'N':     len(true),
        'R²':    r2_score(true, pred),
        'MSE':   mean_squared_error(true, pred),
        'RMSE':  np.sqrt(mean_squared_error(true, pred)),
        'MAE':   mean_absolute_error(true, pred),
        'pred':  pred,
        'true':  true
    }

tr = evaluate(Xtr, y_train, 'Training')
te = evaluate(Xte, y_test,  'Testing')

print('PAPER TABLE 7 — Model Performance')
print('=' * 70)
print(f"{'Dataset':<10} {'N':>5} {'R²':>10} {'MSE':>10} {'RMSE':>10} {'MAE':>10}")
print('-' * 70)
for r in (tr, te):
    print(f"{r['Label']:<10} {r['N']:>5} {r['R²']:>10.4f} {r['MSE']:>10.4f} {r['RMSE']:>10.4f} {r['MAE']:>10.4f}")
print('=' * 70)

## 11. Cross Plots and Fit Plots → Paper Figs. 4 & 5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

for ax, r, color in [
    (axes[0,0], tr, '#1D9E75'),
    (axes[0,1], te, '#D85A30')
]:
    ax.scatter(r['pred'], r['true'], color=color, alpha=0.7,
                edgecolors='k', linewidths=0.4, s=50)
    lims = [min(r['true'].min(), r['pred'].min())-2,
             max(r['true'].max(), r['pred'].max())+2]
    ax.plot(lims, lims, 'k--', lw=1, label='y = x')
    m, b = np.polyfit(r['pred'].ravel(), r['true'].ravel(), 1)
    ax.plot(np.array(lims), m*np.array(lims)+b, color=color, lw=1.5,
             label=f'y = {m:.4f}x + {b:.4f}')
    ax.text(0.05, 0.92, f'R² = {r["R²"]:.4f}', transform=ax.transAxes,
             color=color, fontsize=11, fontweight='bold')
    ax.set_xlabel('BRANN predicted IE%')
    ax.set_ylabel('Experimental IE%')
    ax.set_title(f'Cross plot — {r["Label"]} dataset')
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(True, alpha=0.3)

for ax, r, color in [
    (axes[1,0], tr, '#1D9E75'),
    (axes[1,1], te, '#D85A30')
]:
    idx = np.arange(len(r['true']))
    ax.plot(idx, r['true'].ravel(),  'k-',  lw=1.5, label='Experimental')
    ax.plot(idx, r['pred'].ravel(), '--',   color=color, lw=1.5, label='BRANN predicted')
    ax.set_xlabel('Data index')
    ax.set_ylabel('IE%')
    ax.set_title(f'Fit plot — {r["Label"]} dataset')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('paper_fig_crossplot_fitplot.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: paper_fig_crossplot_fitplot.png')

## 12. SHAP Analysis → Paper Figs. 6a, 6b, 6c

In [ ]:
def predict_for_shap(X_raw):
    X_norm = scaler_X.transform(X_raw)
    with torch.no_grad():
        p = final_model(torch.FloatTensor(X_norm)).numpy()
    return scaler_y.inverse_transform(p).ravel()

X_train_raw = scaler_X.inverse_transform(X_train)
X_test_raw  = scaler_X.inverse_transform(X_test)

explainer = shap.KernelExplainer(predict_for_shap, shap.kmeans(X_train_raw, 10))
shap_values = explainer.shap_values(X_test_raw, nsamples=200)
print(f'SHAP values computed. Shape: {np.array(shap_values).shape}')

In [ ]:
# 12a. Bar chart — mean |SHAP|
fig, ax = plt.subplots(figsize=(8, 5))
mean_abs_shap = np.abs(shap_values).mean(axis=0)
signed_mean   = np.array(shap_values).mean(axis=0)
order = np.argsort(mean_abs_shap)
colors = ['#1D9E75' if signed_mean[i] >= 0 else '#D85A30' for i in order]
bars = ax.barh([INPUT_LABELS[i] for i in order],
                mean_abs_shap[order],
                color=colors, edgecolor='k', linewidth=0.5)
ax.set_xlabel('Mean |SHAP value| (average impact on IE%)')
ax.set_title('Global SHAP feature importance')
ax.grid(True, axis='x', alpha=0.3)
for bar, val in zip(bars, mean_abs_shap[order]):
    ax.text(val + 0.01*max(mean_abs_shap), bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('paper_fig_shap_bar.png', dpi=300, bbox_inches='tight')
plt.show()

print('SHAP feature ranking (for paper text):')
for i in np.argsort(-mean_abs_shap):
    direction = '+' if signed_mean[i] > 0 else '-'
    print(f'  {INPUT_LABELS[i]:25s}  mean|SHAP| = {mean_abs_shap[i]:.4f}  ({direction})')

In [ ]:
# 12b. Beeswarm
shap_exp = shap.Explanation(
    values=shap_values,
    data=X_test_raw,
    feature_names=INPUT_LABELS
)
plt.figure(figsize=(8, 5))
shap.plots.beeswarm(shap_exp, show=False, max_display=5)
plt.title('SHAP beeswarm — directionality of feature impact on IE%')
plt.tight_layout()
plt.savefig('paper_fig_shap_beeswarm.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 12c. Dependence plot for top feature
top_idx = int(np.argmax(mean_abs_shap))
plt.figure(figsize=(8, 5))
shap.dependence_plot(top_idx, shap_values, X_test_raw,
                       feature_names=INPUT_LABELS, show=False)
plt.title(f'SHAP dependence — {INPUT_LABELS[top_idx]}')
plt.tight_layout()
fname = f'paper_fig_shap_dependence_{INPUT_LABELS[top_idx].replace(" ", "_")}.png'
plt.savefig(fname, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {fname}')

## 13. Williams Plot → Paper Fig. 7 (Applicability Domain)

In [ ]:
final_model.eval()
X_all_norm = scaler_X.transform(df[INPUT_COLS].values)
y_all_norm = scaler_y.transform(df[OUTPUT_COL].values.reshape(-1, 1))

with torch.no_grad():
    pred_all_norm = final_model(torch.FloatTensor(X_all_norm)).numpy()

residuals     = y_all_norm.ravel() - pred_all_norm.ravel()
std_residuals = (residuals - residuals.mean()) / (residuals.std() + 1e-10)

H = X_all_norm @ np.linalg.pinv(X_all_norm.T @ X_all_norm) @ X_all_norm.T
leverage = np.diag(H)

p, n = X_all_norm.shape[1], X_all_norm.shape[0]
H_star = 3 * (p + 1) / n

is_outlier       = np.abs(std_residuals) > 3
is_high_leverage = leverage > H_star
is_valid         = ~is_outlier & ~is_high_leverage

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(leverage[is_valid], std_residuals[is_valid],
            c='#1D9E75', label='Valid', s=50, edgecolors='k', linewidths=0.4, alpha=0.8)
ax.scatter(leverage[is_outlier], std_residuals[is_outlier],
            c='#E24B4A', label='Outlier', s=70, edgecolors='k', linewidths=0.6, zorder=5)
ax.scatter(leverage[is_high_leverage & ~is_outlier],
            std_residuals[is_high_leverage & ~is_outlier],
            c='#EF9F27', label='High leverage', s=70, edgecolors='k', linewidths=0.6, zorder=5)

ax.axhline(3, color='k', ls='--', lw=0.8)
ax.axhline(-3, color='k', ls='--', lw=0.8)
ax.axvline(H_star, color='k', ls='--', lw=0.8)
ax.text(H_star+0.001, ax.get_ylim()[1]*0.85, f'H* = {H_star:.4f}',
         fontsize=10, color='k')
ax.set_xlabel('Leverage values (h)')
ax.set_ylabel('Standardised residuals')
ax.set_title('Williams plot — applicability domain of BRANN model')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('paper_fig_williams_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'H* threshold     : {H_star:.4f}')
print(f'Valid points     : {is_valid.sum()}/{n} ({100*is_valid.sum()/n:.1f}%)')
print(f'Outliers         : {is_outlier.sum()}')
print(f'High leverage    : {is_high_leverage.sum()}')

## 14. Save All Results

In [ ]:
with pd.ExcelWriter('BRANN_paper_results.xlsx') as writer:
    stats.round(4).to_excel(writer, sheet_name='Table3_Statistics')
    corr_df.round(4).to_excel(writer, sheet_name='Heatmap_Correlations')
    pd.DataFrame({'epoch': range(len(history)), 'mse': history}).to_excel(writer,
        sheet_name='Training_Curve', index=False)
    weights_dict = {'Variable': INPUT_LABELS + ['b1', 'LW', 'b2']}
    for j in range(optimal_n_h):
        col = [IW[j, i] for i in range(5)] + [b1[j], LW[0, j], b2[0] if j == 0 else None]
        weights_dict[f'Neuron_{j+1}'] = col
    pd.DataFrame(weights_dict).to_excel(writer, sheet_name='Table6_Weights', index=False)
    pd.DataFrame([
        {'Dataset': 'Training', 'N': tr['N'], 'R²': tr['R²'], 'MSE': tr['MSE'], 'RMSE': tr['RMSE'], 'MAE': tr['MAE']},
        {'Dataset': 'Testing',  'N': te['N'], 'R²': te['R²'], 'MSE': te['MSE'], 'RMSE': te['RMSE'], 'MAE': te['MAE']}
    ]).to_excel(writer, sheet_name='Table7_Performance', index=False)
    pd.DataFrame(shap_values, columns=INPUT_LABELS).to_excel(writer,
        sheet_name='SHAP_Values', index=False)
    pd.DataFrame({
        'Leverage':       leverage,
        'Std_Residual':   std_residuals,
        'Status':         np.where(is_outlier, 'Outlier',
                           np.where(is_high_leverage, 'High Leverage', 'Valid'))
    }).to_excel(writer, sheet_name='Williams_Plot', index=False)

print('All results saved to BRANN_paper_results.xlsx')